# Google Search Ranking & Discoverability Capstone — Refresh / Content Opportunity Scoring

**Lane:** Refresh / Content Opportunity Scoring  
**Decision:** Which existing pages should an editor review first?  
**Public-safe scope:** anonymized starter data only; no client names, domains, URLs, titles, or queries.

## 1. Question

**Research question:** Can observed search visibility, engagement, freshness, and content signals rank pages by refresh-review priority better than a transparent baseline rule?

**Task type:** ranking / scoring.  
**Proxy target:** observed declining trend (`is_declining_label`).  
**Success metric:** Precision@50, supported by ROC AUC and average precision.  
**Action:** give editors a ranked review queue.  
**Why ML:** a fixed rule combines a few thresholds; the learned model can combine many correlated signals and produce a calibrated ordering while remaining a decision-support tool—not a claim about Google's ranking system.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'outputs').exists():
    ROOT = ROOT.parent

REPORT = ROOT / 'outputs' / 'model_report.md'
QUEUE = ROOT / 'outputs' / 'refresh_queue_sample.csv'
SUMMARY = ROOT / 'outputs' / 'summary.json'

print('Repository root:', ROOT)
print('Report exists:', REPORT.exists())
print('Queue sample exists:', QUEUE.exists())

queue = pd.read_csv(QUEUE)
print('Rows in published sample:', len(queue))
display(queue.head(10))

## 2. Data

The repository's bundled pipeline report records **30,000 pages** scored from the anonymized starter dataset. The observed declining-label rate was **54.2% (16,262 / 30,000)**.

The capstone uses the already-generated queue artifact for reproducible inspection because the GitHub connector exposes the CSV path but not its raw bytes in this environment. The repository's `scripts/run_all.py` is the source-of-truth rerunnable pipeline and explicitly expects `data/raw/content_refresh_anonymized.csv`.

Public-safe exclusions: identifiers that could expose clients or pages, plus titles, URLs, domains, and search queries.

In [ ]:
# Unit of analysis: one existing content page.
unit_columns = [
    'content_id', 'impressions_90d', 'clicks_90d', 'sessions_90d',
    'avg_position', 'ctr', 'content_age_days', 'word_count', 'trend_direction'
]
print('One row = one existing content page.')
display(queue[[c for c in unit_columns if c in queue.columns]].head(10))

print('\nTrend distribution in the published queue sample:')
display(queue['trend_direction'].value_counts(dropna=False))

## 3. Methodology

**Validation:** the repository's production-style reference pipeline uses a **client-holdout split**, keeping validation separated by client rather than randomly mixing pages.

**Models:** logistic regression, decision tree, and random forest are compared. The final model is selected by Precision@50.

**Baseline:** a transparent hand-written refresh rule (`baseline_rules`) is evaluated on the same validation setup.

**Leakage check:** page identifiers and other public-unsafe/high-cardinality identifiers are not treated as predictive content signals. The pipeline derives the declining label from the observed trend and evaluates the ranking on held-out data.

**Interpretation:** feature importance is directional association within this dataset, not causation.

In [ ]:
# Measured validation results from outputs/model_report.md
results = pd.DataFrame([
    ['decision_tree', 0.742, 0.575, 0.540, 0.716, 0.634],
    ['logistic_regression', 0.700, 0.522, 0.400, 0.567, 0.566],
    ['random_forest', 0.750, 0.618, 0.740, 0.744, 0.640],
    ['baseline_rules', 0.627, 0.468, 0.240, None, None],
], columns=['model','roc_auc','average_precision','precision_at_50','recall','f1'])
display(results)

best = results.loc[results['precision_at_50'].idxmax()]
print(f"Best model: {best['model']}")
print(f"Precision@50 lift over baseline: {best['precision_at_50'] - results.loc[results.model == 'baseline_rules','precision_at_50'].iloc[0]:.3f} absolute")

## 4. Results (vs baseline)

The measured comparison is on the same client-holdout validation design. The random forest is the selected model: **Precision@50 = 0.740 vs 0.240 for the baseline**, an absolute lift of **0.500** at the top-50 review cutoff. Its ROC AUC is **0.750 vs 0.627** for the baseline.

The result supports prioritization quality on this anonymized sample. It does **not** demonstrate that refreshing a page causes better rankings or traffic.

In [ ]:
# Practical interpretation of the measured queue.
queue['suggested_action'].value_counts().rename_axis('action').to_frame('rows')

## 5. Limitations

1. The label is an observed decline proxy, not a business outcome and not a Google ranking label.
2. The analysis is observational; it cannot prove that refreshing content causes recovery in traffic, rankings, or clicks.
3. The starter release is anonymized and intentionally omits client/page identity.
4. Performance can vary with data refreshes, library versions, and split composition.
5. The queue is decision support. Editors should verify context before changing or publishing content.
6. The model does not reproduce or reverse-engineer Google's ranking algorithm.

## 6. Ranked recommendations

Use the top of the queue as an editorial review order. The generated full queue is `outputs/refresh_queue.csv`; the public-safe preview is `outputs/refresh_queue_sample.csv`.

Recommended workflow:

- **Priority review:** high-confidence rows with visible demand and decline risk.
- **CTR review:** pages with meaningful impressions, strong visibility, and weak CTR signals.
- **Engagement review:** pages with visible sessions and weaker engagement signals.
- **Monitor:** lower-confidence items until new evidence arrives.

The top rows are recommendations for human review—not automatic refresh instructions.

In [ ]:
public_cols = [c for c in [
    'final_rank','content_id','final_refresh_score','best_model_probability',
    'confidence','suggested_action','impressions_90d','sessions_90d',
    'avg_position','ctr','content_age_days','word_count','trend_direction'
] if c in queue.columns]
display(queue[public_cols].head(25))

## 7. Artifacts the paper embeds

The paper is backed by repository artifacts rather than hand-entered claims:

- `outputs/model_report.md` — measured dataset and validation summary
- `outputs/refresh_queue_sample.csv` — public-safe ranked preview
- `outputs/charts/top_feature_importance.svg` — directional feature importance
- `outputs/charts/action_mix.svg` — recommended action mix
- `outputs/charts/confidence_mix.svg` — queue confidence mix
- `paper/assets/model_vs_baseline.svg` — model/baseline comparison chart
- `paper/assets/ranked_recommendations_top25.csv` — public-safe top-25 recommendation table

In [ ]:
from pathlib import Path
assets = [
    ROOT/'outputs/model_report.md',
    ROOT/'outputs/refresh_queue_sample.csv',
    ROOT/'outputs/charts/top_feature_importance.svg',
    ROOT/'paper/assets/model_vs_baseline.svg',
    ROOT/'paper/assets/ranked_recommendations_top25.csv',
]
for p in assets:
    print(('OK  ' if p.exists() else 'MISS '), p.relative_to(ROOT) if p.exists() else p)

## ML-12 — 5-minute demo outline

1. **0:00–0:45:** State the decision: prioritize pages for editorial review.
2. **0:45–1:30:** Show the unit of analysis and the observed-decline proxy label.
3. **1:30–2:30:** Explain the client-holdout validation and baseline.
4. **2:30–3:30:** Show Precision@50: 0.740 model vs 0.240 baseline.
5. **3:30–4:30:** Walk through the ranked queue and reason codes.
6. **4:30–5:00:** Close with limitations and the next test: measure outcomes after controlled editorial changes.

## ML-12 — social-post cut

> Built a refresh-priority scorer on anonymized search data. The selected model reached Precision@50 of 0.74 versus 0.24 for the transparent baseline on the same client-holdout evaluation. The output is a ranked editorial review queue—not a claim about Google's algorithm or a causal proof that refreshes recover traffic.

## ML-12 — employer-facing summary

I built a search-content refresh prioritization workflow that turns page-level search and content signals into an explainable review queue. On the anonymized starter release, the selected model improved Precision@50 from 0.24 for the baseline to 0.74 on a client-holdout evaluation. I framed the result as decision support, documented leakage/validation limits, and shipped reproducible artifacts rather than overstating causal or Google-algorithm claims.

## Self-check

- [x] Question, task type, proxy target, metric, action, and ML rationale are explicit.
- [x] Unit of analysis is shown as a dataframe.
- [x] Model vs baseline is measured on the same validation design.
- [x] Limitations and public-safe framing are explicit.
- [x] Ranked recommendations and reproducibility artifacts are named.
- [x] ML-12 demo, social cut, and employer summary are included.

**Run locally:** `python scripts/run_all.py` after restoring the bundled anonymized CSV if needed.